In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import nltk
from nltk.corpus import stopwords
import unicodedata

ruta_procesado = "../data/processed/tickets_etiquetados.csv"
df = pd.read_csv(ruta_procesado)
df['descripcion'] = df['descripcion'].fillna('')

# Estandarizar texto quitando tildes
def quitar_tildes(texto):
    return unicodedata.normalize('NFD', texto).encode('ascii', 'ignore').decode("utf-8")

df['desc_limpia'] = df['descripcion'].apply(quitar_tildes)

# Limpiar tildes también en las stopwords
stop_words_es = [quitar_tildes(w) for w in stopwords.words('spanish')]

# Configurar TF-IDF (ahora sin vocales acentuadas en el regex)
vectorizador = TfidfVectorizer(
    max_features=1000, 
    stop_words=stop_words_es,
    token_pattern=r'(?u)\b[a-zA-ZñÑ]+\b' 
)

# Definir variables predictoras (X) y objetivo (y)
X = vectorizador.fit_transform(df['desc_limpia'])
y = df['target_sla']

# Split Train/Test manteniendo la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Vocabulario limpio. Ejemplo: {vectorizador.get_feature_names_out()[140:150]}")
print(f"Muestras de entrenamiento: {X_train.shape[0]}")
print(f"Muestras de test: {X_test.shape[0]}")

Vocabulario limpio. Ejemplo: ['caerse' 'caida' 'caidas' 'caido' 'caja' 'cajas' 'cajero' 'cajeros'
 'cajetin' 'cajon']
Muestras de entrenamiento: 1128
Muestras de test: 283


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Instanciar modelo forzando el balanceo de clases
modelo_lr = LogisticRegression(class_weight='balanced', random_state=42)

# Entrenar modelo con los datos de Train
modelo_lr.fit(X_train, y_train)

# Generar predicciones sobre los datos de Test
y_pred = modelo_lr.predict(X_test)

# Evaluar rendimiento
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

Matriz de Confusión:
[[244  25]
 [ 12   2]]

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.95      0.91      0.93       269
           1       0.07      0.14      0.10        14

    accuracy                           0.87       283
   macro avg       0.51      0.52      0.51       283
weighted avg       0.91      0.87      0.89       283



In [6]:
import pandas as pd

# Extraer los coeficientes matemáticos de la regresión
coeficientes = modelo_lr.coef_[0]
palabras = vectorizador.get_feature_names_out()

df_importancia = pd.DataFrame({'Palabra': palabras, 'Peso': coeficientes})

print("Top 10 palabras asociadas a RETRASOS (> 21 días):")
print(df_importancia.sort_values(by='Peso', ascending=False).head(10).to_string(index=False))

print("\nTop 10 palabras asociadas a CIERRE RÁPIDO:")
print(df_importancia.sort_values(by='Peso', ascending=True).head(10).to_string(index=False))

Top 10 palabras asociadas a RETRASOS (> 21 días):
       Palabra     Peso
   emergencias 3.433690
    descolgada 2.752245
        chicas 2.385080
    correctivo 1.992938
        varias 1.980232
       cortina 1.950695
          sale 1.881058
frecuentemente 1.845300
       cordoba 1.795925
          rota 1.782527

Top 10 palabras asociadas a CIERRE RÁPIDO:
      Palabra      Peso
         roto -1.409206
acondicionado -1.379391
  necesitamos -1.345044
     cisterna -1.301556
         mesa -1.215590
     limpieza -1.130370
      cambiar -1.097853
      revisar -1.060076
        grifo -1.047955
      mujeres -0.990194


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Limpiar nulos en la nueva variable
df['oficina'] = df['oficina'].fillna('Sin_Oficina').astype(str)

# Construir el procesador de columnas
procesador_hibrido = ColumnTransformer(
    transformers=[
        ('texto', TfidfVectorizer(max_features=1000, stop_words=stop_words_es, token_pattern=r'(?u)\b[a-zA-ZñÑ]+\b'), 'desc_limpia'),
        ('metadata', OneHotEncoder(handle_unknown='ignore'), ['oficina'])
    ])

# Ensamblar el Pipeline 
pipeline_lr = Pipeline(steps=[
    ('preprocesador', procesador_hibrido),
    ('algoritmo', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

# Redefinir X incluyendo la oficina y dividir datos
X_multi = df[['desc_limpia', 'oficina']]
y_multi = df['target_sla']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

# Entrenar y evaluar
pipeline_lr.fit(X_train_m, y_train_m)
y_pred_m = pipeline_lr.predict(X_test_m)

print("Matriz de Confusión (Texto + Oficina):")
print(confusion_matrix(y_test_m, y_pred_m))
print("\nReporte de Clasificación (Texto + Oficina):")
print(classification_report(y_test_m, y_pred_m))

Matriz de Confusión (Texto + Oficina):
[[240  29]
 [  6   8]]

Reporte de Clasificación (Texto + Oficina):
              precision    recall  f1-score   support

           0       0.98      0.89      0.93       269
           1       0.22      0.57      0.31        14

    accuracy                           0.88       283
   macro avg       0.60      0.73      0.62       283
weighted avg       0.94      0.88      0.90       283



In [8]:
from sklearn.ensemble import RandomForestClassifier

# Mantenemos el mismo procesador pero cambiando el motor
pipeline_rf = Pipeline(steps=[
    ('preprocesador', procesador_hibrido),
    # Usamos 100 árboles y forzamos el balanceo de clases
    ('algoritmo', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

# Entrenamiento del Random Forest
pipeline_rf.fit(X_train_m, y_train_m)

# Evaluación de su rendimiento
y_pred_rf = pipeline_rf.predict(X_test_m)

print("Matriz de Confusión (Random Forest):")
print(confusion_matrix(y_test_m, y_pred_rf))
print("\nReporte de Clasificación (Random Forest):")
print(classification_report(y_test_m, y_pred_rf))

Matriz de Confusión (Random Forest):
[[265   4]
 [ 14   0]]

Reporte de Clasificación (Random Forest):
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       269
           1       0.00      0.00      0.00        14

    accuracy                           0.94       283
   macro avg       0.47      0.49      0.48       283
weighted avg       0.90      0.94      0.92       283



In [9]:
import joblib
import os

# Crear la carpeta models
os.makedirs("../models", exist_ok=True)

# Guardar el pipeline ganador (Regresión Logística)
ruta_modelo = "../models/modelo_sla_logistico.pkl"
joblib.dump(pipeline_lr, ruta_modelo)

print(f"Modelo ganador serializado y guardado con éxito en: {ruta_modelo}")

Modelo ganador serializado y guardado con éxito en: ../models/modelo_sla_logistico.pkl
